In [0]:
# Lendo o CSV da camada Bronze 
df_bronze = spark.read.csv(
    "abfss://bronze@stmedallionmiguel.dfs.core.windows.net/sales.csv",
    header=True,
    inferSchema=True
    )
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- SalesOrderNumber: string (nullable = true)
 |-- SalesOrderLineNumber: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- TaxAmount: double (nullable = true)

+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+---------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|  CustomerName|        EmailAddress|                Item|Quantity|UnitPrice|TaxAmount|
+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+---------+
|         SO43701|                   1|2019-07-01|   Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99| 271.9992|
|         SO43704|                   1|2019-07-01|    Julio

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

df_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns
]).show()

+----------------+--------------------+---------+------------+------------+----+--------+---------+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate|CustomerName|EmailAddress|Item|Quantity|UnitPrice|TaxAmount|
+----------------+--------------------+---------+------------+------------+----+--------+---------+---------+
|               0|                   0|        0|           0|           0|   0|       0|        0|        0|
+----------------+--------------------+---------+------------+------------+----+--------+---------+---------+



In [0]:
# Duplicatas totais (linha inteira repetida)
total_rows = df_bronze.count()
distinct_rows = df_bronze.distinct().count()
print(f"Total de linhas: {total_rows}")
print(f"LInhas distintas: {distinct_rows}")
print(f"Duplicatas totais: {total_rows - distinct_rows}")

# Duplicatas na chave lógica(pedido + item do pedido)
from pyspark.sql.functions import count

df_bronze.groupBy("SalesOrderNumber", "SalesOrderLineNumber") \
.agg(count("*").alias("qtd")) \
.filter("qtd > 1") \
.show()

Total de linhas: 32718
LInhas distintas: 32718
Duplicatas totais: 0
+----------------+--------------------+---+
|SalesOrderNumber|SalesOrderLineNumber|qtd|
+----------------+--------------------+---+
+----------------+--------------------+---+



In [0]:
df_silver = (
    df_bronze
    .withColumnRenamed("SalesOrderNumber", "sales_order_number")
    .withColumnRenamed("SalesOrderLineNumber", "sales_order_line_number")
    .withColumnRenamed("OrderDate", "order_date")
    .withColumnRenamed("CustomerName", "customer_name")
    .withColumnRenamed("EmailAddress", "email_address")
    .withColumnRenamed("Item", "item")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("UnitPrice", "unit_price")
    .withColumnRenamed("TaxAmount", "tax_amount")
)

df_silver.printSchema()
df_silver.show(5)

root
 |-- sales_order_number: string (nullable = true)
 |-- sales_order_line_number: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email_address: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- tax_amount: double (nullable = true)

+------------------+-----------------------+----------+--------------+--------------------+--------------------+--------+----------+----------+
|sales_order_number|sales_order_line_number|order_date| customer_name|       email_address|                item|quantity|unit_price|tax_amount|
+------------------+-----------------------+----------+--------------+--------------------+--------------------+--------+----------+----------+
|           SO43701|                      1|2019-07-01|   Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|   3399.99|  271.9992|
|           SO43704| 

In [0]:
from pyspark.sql.functions import lower, trim

df_silver = (
    df_silver
    .withColumn("email_address", lower(trim(df_silver.email_address)))
    .withColumn("customer_name", trim(df_silver.customer_name))
    .withColumn("item", trim(df_silver.item))
)

df_silver.show(5, truncate=False)

+------------------+-----------------------+----------+--------------+-----------------------------+-----------------------+--------+----------+----------+
|sales_order_number|sales_order_line_number|order_date|customer_name |email_address                |item                   |quantity|unit_price|tax_amount|
+------------------+-----------------------+----------+--------------+-----------------------------+-----------------------+--------+----------+----------+
|SO43701           |1                      |2019-07-01|Christy Zhu   |christy12@adventure-works.com|Mountain-100 Silver, 44|1       |3399.99   |271.9992  |
|SO43704           |1                      |2019-07-01|Julio Ruiz    |julio1@adventure-works.com   |Mountain-100 Black, 48 |1       |3374.99   |269.9992  |
|SO43705           |1                      |2019-07-01|Curtis Lu     |curtis9@adventure-works.com  |Mountain-100 Silver, 38|1       |3399.99   |271.9992  |
|SO43700           |1                      |2019-07-01|Ruben Pra

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .save("abfss://silver@stmedallionmiguel.dfs.core.windows.net/sales")
)

print("Escrita concluída na camada Silver.")

Escrita concluída na camada Silver.


In [0]:
# Ver o histórico de versões da tabela Delta
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "abfss://silver@stmedallionmiguel.dfs.core.windows.net/sales")
delta_table.history().show(truncate=False)